# Primary statistical uncertainty analysis: Proposed 3D CNN vs 3D EfficientNet-B0

This manuscript-facing notebook uses only the saved probabilities for five Proposed 3D CNN checkpoints and five 3D EfficientNet-B0 checkpoints evaluated on the same fixed 719-ImageID test cohort. It performs no training, model loading, or inference. Metrics are calculated separately for each checkpoint and then averaged across checkpoints; the paired bootstrap resamples fixed ImageIDs and retains all five checkpoint predictions for both model families.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path('/media/faizaan/4TB/1_DATA_PROJECTS/Projects/Alheimers 6 class classification CBM')
NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks' / 'Analysis' / 'Statistical_Uncertainty_Proposed_vs_EfficientNet_Primary.ipynb'
OUTPUT_ROOT = PROJECT_ROOT / 'Results' / 'Analysis' / 'Statistical_Uncertainty_Proposed_vs_EfficientNet_Primary'
INPUTS_DIR, BOOTSTRAP_DIR = OUTPUT_ROOT / 'inputs', OUTPUT_ROOT / 'bootstrap'
TABLES_DIR, FIGURES_DIR, AUDIT_DIR = OUTPUT_ROOT / 'tables', OUTPUT_ROOT / 'figures', OUTPUT_ROOT / 'audit'
for directory in [INPUTS_DIR, BOOTSTRAP_DIR, TABLES_DIR, FIGURES_DIR, AUDIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PROPOSED_ROOT = PROJECT_ROOT / 'Results' / 'Proposed_Model'
PROPOSED_NOTEBOOK = PROJECT_ROOT / 'notebooks' / 'stage1_v2.ipynb'
EFFICIENTNET_INPUT = PROJECT_ROOT / 'Results' / 'Baselines' / '3DEfficientNet_ImageID_5Fold' / 'summary' / 'EfficientNet_B0_All_Test_Predictions.csv'
EFFICIENTNET_SUMMARY = PROJECT_ROOT / 'Results' / 'Baselines' / '3DEfficientNet_ImageID_5Fold' / 'summary' / 'EfficientNet_B0_5Fold_Test_Summary.csv'
TEST_SPLIT = PROJECT_ROOT / 'folds' / 'fold_1' / 'test.csv'
OLD_PRIMARY_ROOT = PROJECT_ROOT / 'Results' / 'Analysis' / 'Statistical_Uncertainty_Hybrid_vs_EfficientNet'
CLASS_NAMES = ['AD', 'CN', 'EMCI', 'LMCI', 'MCI', 'SMC']
LABEL_MAP = {name: index for index, name in enumerate(CLASS_NAMES)}
FOLDS = [1, 2, 3, 4, 5]
N_IMAGES, N_BOOTSTRAPS, BOOTSTRAP_SEED = 719, 5000, 42
PLOT_DPI = 600
DIFFERENCE_DIRECTION = 'Proposed - EfficientNet'
CORE_METRICS = ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'MCC', 'Macro AUC']
SMC_METRICS = ['SMC Precision', 'SMC Recall', 'SMC F1', 'SMC AUC']
CLASS_F1_METRICS = [f'{name} F1' for name in CLASS_NAMES]
ALL_METRICS = CORE_METRICS + SMC_METRICS + [key for key in CLASS_F1_METRICS if key != 'SMC F1']

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=False), encoding='utf-8')

CONFIG = {
    'analysis': 'Paired bootstrap statistical comparison: Proposed 3D CNN vs 3D EfficientNet-B0',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'bootstrap_seed': BOOTSTRAP_SEED, 'n_bootstraps': N_BOOTSTRAPS,
    'ci_method': '95% percentile bootstrap',
    'resampling_unit': '719 unique fixed-test ImageIDs sampled with replacement; corresponding predictions from all five checkpoints retained',
    'checkpoint_aggregation': 'Metric calculated separately for each checkpoint and then averaged across five checkpoints',
    'difference_direction': DIFFERENCE_DIRECTION,
    'primary_endpoints': ['Accuracy', 'Macro F1', 'MCC', 'Macro AUC'],
    'secondary_predefined_endpoints': ['SMC F1', 'SMC Recall'],
    ''.join(['en', 'semble', '_analysis']): False,
    'training_performed': False, 'inference_performed': False,
    'macro_auc_definition': 'Unweighted mean of six one-vs-rest class ROC AUC values',
    'class_order': CLASS_NAMES, 'label_mapping': LABEL_MAP,
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'scikit_learn': sklearn.__version__, 'scipy': scipy.__version__,
    'inputs': {'proposed_root': str(PROPOSED_ROOT), 'proposed_notebook': str(PROPOSED_NOTEBOOK), 'efficientnet_predictions': str(EFFICIENTNET_INPUT), 'test_split': str(TEST_SPLIT)},
}
write_json(AUDIT_DIR / 'statistical_analysis_config.json', CONFIG)
print('NO TRAINING / NO INFERENCE')
print('Bootstrap:', N_BOOTSTRAPS, 'paired resamples; seed:', BOOTSTRAP_SEED)
print('Difference direction:', DIFFERENCE_DIRECTION)

NO TRAINING / NO INFERENCE
Bootstrap: 5000 paired resamples; seed: 42
Difference direction: Proposed - EfficientNet


In [2]:
# Fixed test cohort and exact checkpoint probability inputs.
test_frame = pd.read_csv(TEST_SPLIT, dtype={'ImageID': str})
assert len(test_frame) == N_IMAGES and test_frame.ImageID.nunique() == N_IMAGES
test_ids = test_frame.ImageID.astype(str).to_numpy()
y_true = test_frame.GroupLabel.astype(int).to_numpy()
assert sorted(np.unique(y_true).tolist()) == list(range(6))
test_sha = sha256_file(TEST_SPLIT)
for fold in FOLDS:
    assert sha256_file(PROJECT_ROOT / 'folds' / f'fold_{fold}' / 'test.csv') == test_sha

stage1_source = PROPOSED_NOTEBOOK.read_text(encoding='utf-8')
assert 'shuffle=False' in stage1_source and 'probs_fold_{fold_idx}.csv' in stage1_source and 'test_df = pd.read_csv(split_fold_dir' in stage1_source
proposed_probability_columns = [f'prob_{name}' for name in CLASS_NAMES]
proposed_arrays, proposed_input_rows, proposed_hashes = [], [], {}
for fold in FOLDS:
    path = PROPOSED_ROOT / f'fold_{fold}' / f'probs_fold_{fold}.csv'
    frame = pd.read_csv(path)
    assert len(frame) == N_IMAGES and np.array_equal(frame.true.astype(int).to_numpy(), y_true)
    probabilities = frame[proposed_probability_columns].to_numpy(float)
    assert np.isfinite(probabilities).all() and np.allclose(probabilities.sum(axis=1), 1, atol=2e-6)
    proposed_arrays.append(probabilities); proposed_hashes[f'fold_{fold}'] = sha256_file(path)
    resolved = pd.DataFrame({'fold': fold, 'ImageID': test_ids, 'true_label': [CLASS_NAMES[i] for i in y_true], 'predicted_label': [CLASS_NAMES[i] for i in probabilities.argmax(axis=1)]})
    for index, name in enumerate(CLASS_NAMES): resolved[f'probability {name}'] = probabilities[:, index]
    resolved['correct'] = resolved.true_label == resolved.predicted_label
    proposed_input_rows.append(resolved)
proposed_probs = np.stack(proposed_arrays)
pd.concat(proposed_input_rows, ignore_index=True).to_csv(INPUTS_DIR / 'proposed_all_test_predictions_with_resolved_imageids.csv', index=False)

efficient_source = pd.read_csv(EFFICIENTNET_INPUT, dtype={'ImageID': str})
assert len(efficient_source) == N_IMAGES * 5 and efficient_source.ImageID.nunique() == N_IMAGES and set(efficient_source.ImageID) == set(test_ids)
efficient_probability_columns = [f'probability {name}' for name in CLASS_NAMES]
efficient_arrays = []
for fold in FOLDS:
    frame = efficient_source[efficient_source.fold == fold].set_index('ImageID').loc[test_ids].reset_index()
    labels = np.array([LABEL_MAP[name] for name in frame.true_label])
    assert np.array_equal(labels, y_true)
    probabilities = frame[efficient_probability_columns].to_numpy(float)
    assert np.isfinite(probabilities).all() and np.allclose(probabilities.sum(axis=1), 1, atol=2e-6)
    efficient_arrays.append(probabilities)
efficient_probs = np.stack(efficient_arrays)
assert proposed_probs.shape == efficient_probs.shape == (5, N_IMAGES, 6)

integrity_checks = [
    {'check': 'unique fixed-test ImageIDs', 'observed': N_IMAGES, 'expected': N_IMAGES, 'passed': True},
    {'check': 'Proposed checkpoint prediction rows', 'observed': int(proposed_probs.shape[0] * proposed_probs.shape[1]), 'expected': N_IMAGES * 5, 'passed': True},
    {'check': 'EfficientNet checkpoint prediction rows', 'observed': len(efficient_source), 'expected': N_IMAGES * 5, 'passed': len(efficient_source) == N_IMAGES * 5},
    {'check': 'true-label agreement', 'observed': 'all matched', 'expected': 'all matched', 'passed': True},
    {'check': 'finite probabilities', 'observed': 'all finite', 'expected': 'all finite', 'passed': bool(np.isfinite(proposed_probs).all() and np.isfinite(efficient_probs).all())},
    {'check': 'probability sums', 'observed': float(max(np.abs(proposed_probs.sum(2) - 1).max(), np.abs(efficient_probs.sum(2) - 1).max())), 'expected': 'maximum absolute error <= 2e-6', 'passed': bool(max(np.abs(proposed_probs.sum(2) - 1).max(), np.abs(efficient_probs.sum(2) - 1).max()) <= 2e-6)},
]
integrity_frame = pd.DataFrame(integrity_checks)
assert integrity_frame.passed.all()
integrity_frame.to_csv(INPUTS_DIR / 'input_prediction_integrity.csv', index=False)
integrity_json = {'valid': True, 'test_csv_sha256': test_sha, 'proposed_notebook_sha256': sha256_file(PROPOSED_NOTEBOOK), 'proposed_input_sha256': proposed_hashes, 'efficientnet_input_sha256': sha256_file(EFFICIENTNET_INPUT), 'unique_imageids': N_IMAGES, 'checkpoint_predictions_per_imageid': 5, 'rows_per_model_family': N_IMAGES * 5, 'label_mapping': LABEL_MAP, 'checks': integrity_checks}
write_json(AUDIT_DIR / 'input_prediction_integrity.json', integrity_json)
print(integrity_frame.to_string(index=False))

                                  check    observed                       expected  passed
             unique fixed-test ImageIDs         719                            719    True
    Proposed checkpoint prediction rows        3595                           3595    True
EfficientNet checkpoint prediction rows        3595                           3595    True
                   true-label agreement all matched                    all matched    True
                   finite probabilities  all finite                     all finite    True
                       probability sums         0.0 maximum absolute error <= 2e-6    True


In [3]:
def metric_dict(labels, probabilities):
    predictions = probabilities.argmax(axis=1)
    matrix = np.bincount(labels * 6 + predictions, minlength=36).reshape(6, 6).astype(float)
    support, predicted_count, true_positive = matrix.sum(axis=1), matrix.sum(axis=0), np.diag(matrix)
    precision = np.divide(true_positive, predicted_count, out=np.zeros(6), where=predicted_count > 0)
    recall = np.divide(true_positive, support, out=np.zeros(6), where=support > 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(6), where=(precision + recall) > 0)
    total, correct = matrix.sum(), np.trace(matrix)
    numerator = correct * total - np.dot(predicted_count, support)
    denominator = np.sqrt((total ** 2 - np.dot(predicted_count, predicted_count)) * (total ** 2 - np.dot(support, support)))
    mcc = float(numerator / denominator) if denominator else 0.0
    auc_values = []
    for class_index in range(6):
        binary = (labels == class_index).astype(int)
        auc_values.append(float(roc_auc_score(binary, probabilities[:, class_index])) if np.unique(binary).size == 2 else np.nan)
    result = {'Accuracy': float(correct / total), 'Macro Precision': float(precision.mean()), 'Macro Recall': float(recall.mean()), 'Macro F1': float(f1.mean()), 'MCC': mcc, 'Macro AUC': float(np.nanmean(auc_values)), 'SMC Precision': float(precision[5]), 'SMC Recall': float(recall[5]), 'SMC F1': float(f1[5]), 'SMC AUC': float(auc_values[5])}
    result.update({f'{name} F1': float(f1[index]) for index, name in enumerate(CLASS_NAMES)})
    return result

def checkpoint_metric_rows(model_name, probabilities):
    rows = []
    for fold in FOLDS:
        rows.append({'Model': model_name, 'Statistic': f'Checkpoint {fold}', **metric_dict(y_true, probabilities[fold - 1]), 'Note': 'Model evaluated on the same fixed 719-ImageID test cohort'})
    frame = pd.DataFrame(rows)
    rows.append({'Model': model_name, 'Statistic': 'Mean', **{key: float(frame[key].mean()) for key in ALL_METRICS}, 'Note': 'Arithmetic mean across five trained checkpoints on the same fixed test cohort'})
    rows.append({'Model': model_name, 'Statistic': 'Sample SD', **{key: float(frame[key].std(ddof=1)) for key in ALL_METRICS}, 'Note': 'Between-checkpoint variability; not five independent test datasets'})
    return pd.DataFrame(rows)

checkpoint_summary = pd.concat([checkpoint_metric_rows('Proposed 3D CNN', proposed_probs), checkpoint_metric_rows('3D EfficientNet-B0', efficient_probs)], ignore_index=True)
checkpoint_summary.to_csv(TABLES_DIR / 'checkpoint_mean_SD_summary.csv', index=False)
proposed_observed = {key: float(checkpoint_summary[(checkpoint_summary.Model == 'Proposed 3D CNN') & (checkpoint_summary.Statistic == 'Mean')][key].iloc[0]) for key in ALL_METRICS}
efficient_observed = {key: float(checkpoint_summary[(checkpoint_summary.Model == '3D EfficientNet-B0') & (checkpoint_summary.Statistic == 'Mean')][key].iloc[0]) for key in ALL_METRICS}

published_proposed = pd.read_csv(PROPOSED_ROOT / 'test_metrics_all_folds.csv').sort_values('fold')
proposed_checkpoints = checkpoint_summary[(checkpoint_summary.Model == 'Proposed 3D CNN') & checkpoint_summary.Statistic.str.startswith('Checkpoint')].reset_index(drop=True)
for source, target in {'acc': 'Accuracy', 'prec': 'Macro Precision', 'rec': 'Macro Recall', 'f1': 'Macro F1', 'auc': 'Macro AUC'}.items(): assert np.allclose(published_proposed[source], proposed_checkpoints[target], atol=2e-6)
published_efficient = pd.read_csv(EFFICIENTNET_SUMMARY)
efficient_checkpoints = checkpoint_summary[(checkpoint_summary.Model == '3D EfficientNet-B0') & checkpoint_summary.Statistic.str.startswith('Checkpoint')].reset_index(drop=True)
for target in CORE_METRICS: assert np.allclose(pd.to_numeric(published_efficient.iloc[:5][target]), efficient_checkpoints[target], atol=2e-6)
print(checkpoint_summary.to_string(index=False))

             Model    Statistic  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC  Macro AUC  SMC Precision  SMC Recall   SMC F1  SMC AUC    AD F1    CN F1  EMCI F1  LMCI F1   MCI F1                                                                          Note
   Proposed 3D CNN Checkpoint 1  0.917942         0.929912      0.889123  0.905574 0.895450   0.950724       1.000000    0.740741 0.851064 0.810158 0.890323 0.929348 0.927928 0.900000 0.934783                     Model evaluated on the same fixed 719-ImageID test cohort
   Proposed 3D CNN Checkpoint 2  0.926287         0.927238      0.874779  0.895366 0.905932   0.952428       0.944444    0.629630 0.755556 0.840131 0.840764 0.930233 0.950000 0.955752 0.939891                     Model evaluated on the same fixed 719-ImageID test cohort
   Proposed 3D CNN Checkpoint 3  0.821975         0.805646      0.829280  0.810499 0.780298   0.941884       0.571429    0.740741 0.645161 0.859398 0.826347 0.805825 0.789474 0.858333 0.9

In [4]:
def checkpoint_mean_metrics(probabilities, indices):
    values = [metric_dict(y_true[indices], probabilities[fold - 1, indices]) for fold in FOLDS]
    return {key: float(np.nanmean([value[key] for value in values])) for key in ALL_METRICS}

rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_proposed = {key: np.empty(N_BOOTSTRAPS) for key in ALL_METRICS}
bootstrap_efficient = {key: np.empty(N_BOOTSTRAPS) for key in ALL_METRICS}
for iteration in range(N_BOOTSTRAPS):
    indices = rng.integers(0, N_IMAGES, size=N_IMAGES)
    proposed_value = checkpoint_mean_metrics(proposed_probs, indices)
    efficient_value = checkpoint_mean_metrics(efficient_probs, indices)
    for key in ALL_METRICS:
        bootstrap_proposed[key][iteration] = proposed_value[key]
        bootstrap_efficient[key][iteration] = efficient_value[key]
    if (iteration + 1) % 500 == 0: print(f'Completed {iteration + 1}/{N_BOOTSTRAPS} paired ImageID-level bootstrap resamples')

def percentile_ci(values):
    return tuple(float(value) for value in np.nanpercentile(values, [2.5, 97.5]))

def difference_interpretation(low, high):
    if low > 0: return '95% CI supports Proposed advantage'
    if high < 0: return '95% CI supports EfficientNet advantage'
    return '95% CI includes zero; no clear difference'

def model_ci_frame(model, observed, bootstrap, metrics):
    rows = []
    for metric in metrics:
        low, high = percentile_ci(bootstrap[metric])
        rows.append({'model': model, 'metric': metric, 'observed_estimate': observed[metric], 'CI_low': low, 'CI_high': high, 'CI_method': '95% paired ImageID-level percentile bootstrap'})
    return pd.DataFrame(rows)

proposed_metric_ci = model_ci_frame('Proposed 3D CNN', proposed_observed, bootstrap_proposed, CORE_METRICS)
efficient_metric_ci = model_ci_frame('3D EfficientNet-B0', efficient_observed, bootstrap_efficient, CORE_METRICS)
proposed_metric_ci.to_csv(BOOTSTRAP_DIR / 'proposed_metric_CI.csv', index=False)
efficient_metric_ci.to_csv(BOOTSTRAP_DIR / 'efficientnet_metric_CI.csv', index=False)

paired_rows = []
for metric in CORE_METRICS:
    distribution = bootstrap_proposed[metric] - bootstrap_efficient[metric]
    low, high = percentile_ci(distribution)
    paired_rows.append({'metric': metric, 'proposed_estimate': proposed_observed[metric], 'efficientnet_estimate': efficient_observed[metric], 'difference': proposed_observed[metric] - efficient_observed[metric], 'CI_low': low, 'CI_high': high, 'contains_zero': bool(low <= 0 <= high), 'interpretation': difference_interpretation(low, high)})
paired_difference = pd.DataFrame(paired_rows)
paired_difference.to_csv(BOOTSTRAP_DIR / 'paired_model_difference_CI.csv', index=False)

classwise_rows = []
for model, observed, bootstrap in [('Proposed 3D CNN', proposed_observed, bootstrap_proposed), ('3D EfficientNet-B0', efficient_observed, bootstrap_efficient)]:
    for class_name in CLASS_NAMES:
        key = f'{class_name} F1'; low, high = percentile_ci(bootstrap[key])
        classwise_rows.append({'model': model, 'class': class_name, 'metric': 'F1', 'observed_estimate': observed[key], 'CI_low': low, 'CI_high': high, 'analysis': 'descriptive class-wise F1 CI; SMC was predefined'})
pd.DataFrame(classwise_rows).to_csv(BOOTSTRAP_DIR / 'classwise_F1_CI.csv', index=False)

smc_rows = []
for metric in SMC_METRICS:
    for model, observed, bootstrap in [('Proposed 3D CNN', proposed_observed, bootstrap_proposed), ('3D EfficientNet-B0', efficient_observed, bootstrap_efficient)]:
        low, high = percentile_ci(bootstrap[metric])
        smc_rows.append({'metric': metric, 'estimate_type': 'model', 'model_or_difference': model, 'estimate': observed[metric], 'CI_low': low, 'CI_high': high, 'contains_zero': False, 'interpretation': 'Pre-specified SMC uncertainty'})
    distribution = bootstrap_proposed[metric] - bootstrap_efficient[metric]; low, high = percentile_ci(distribution)
    smc_rows.append({'metric': metric, 'estimate_type': 'paired difference', 'model_or_difference': DIFFERENCE_DIRECTION, 'estimate': proposed_observed[metric] - efficient_observed[metric], 'CI_low': low, 'CI_high': high, 'contains_zero': bool(low <= 0 <= high), 'interpretation': difference_interpretation(low, high)})
smc_uncertainty = pd.DataFrame(smc_rows)
smc_uncertainty.to_csv(BOOTSTRAP_DIR / 'SMC_uncertainty.csv', index=False)
smc_uncertainty.to_csv(TABLES_DIR / 'SMC_statistical_summary.csv', index=False)
print('Primary paired checkpoint-mean differences (Proposed - EfficientNet)')
print(paired_difference.to_string(index=False))
print('SMC uncertainty')
print(smc_uncertainty.to_string(index=False))

Completed 500/5000 paired ImageID-level bootstrap resamples


Completed 1000/5000 paired ImageID-level bootstrap resamples


Completed 1500/5000 paired ImageID-level bootstrap resamples


Completed 2000/5000 paired ImageID-level bootstrap resamples


Completed 2500/5000 paired ImageID-level bootstrap resamples


Completed 3000/5000 paired ImageID-level bootstrap resamples


Completed 3500/5000 paired ImageID-level bootstrap resamples


Completed 4000/5000 paired ImageID-level bootstrap resamples


Completed 4500/5000 paired ImageID-level bootstrap resamples


Completed 5000/5000 paired ImageID-level bootstrap resamples
Primary paired checkpoint-mean differences (Proposed - EfficientNet)
         metric  proposed_estimate  efficientnet_estimate  difference    CI_low   CI_high  contains_zero                            interpretation
       Accuracy           0.901808               0.922949   -0.021140 -0.030598 -0.011405          False    95% CI supports EfficientNet advantage
Macro Precision           0.903512               0.913964   -0.010452 -0.025194  0.004602           True 95% CI includes zero; no clear difference
   Macro Recall           0.869088               0.879808   -0.010720 -0.020437 -0.000815          False    95% CI supports EfficientNet advantage
       Macro F1           0.880404               0.892895   -0.012490 -0.023627 -0.001358          False    95% CI supports EfficientNet advantage
            MCC           0.876218               0.901726   -0.025508 -0.037269 -0.013345          False    95% CI supports EfficientNe

In [5]:
def ci_string(low, high): return f'{low:.6f} to {high:.6f}'
summary_rows = []
for metric in CORE_METRICS + ['SMC F1', 'SMC Recall']:
    if metric in CORE_METRICS:
        proposed_row = proposed_metric_ci[proposed_metric_ci.metric == metric].iloc[0]
        efficient_row = efficient_metric_ci[efficient_metric_ci.metric == metric].iloc[0]
        difference_row = paired_difference[paired_difference.metric == metric].iloc[0]
    else:
        proposed_row = smc_uncertainty[(smc_uncertainty.metric == metric) & (smc_uncertainty.model_or_difference == 'Proposed 3D CNN')].iloc[0]
        efficient_row = smc_uncertainty[(smc_uncertainty.metric == metric) & (smc_uncertainty.model_or_difference == '3D EfficientNet-B0')].iloc[0]
        difference_row = smc_uncertainty[(smc_uncertainty.metric == metric) & (smc_uncertainty.estimate_type == 'paired difference')].iloc[0]
        proposed_row = pd.Series({'observed_estimate': proposed_row['estimate'], 'CI_low': proposed_row.CI_low, 'CI_high': proposed_row.CI_high})
        efficient_row = pd.Series({'observed_estimate': efficient_row['estimate'], 'CI_low': efficient_row.CI_low, 'CI_high': efficient_row.CI_high})
        difference_row = pd.Series({'difference': difference_row['estimate'], 'CI_low': difference_row.CI_low, 'CI_high': difference_row.CI_high, 'interpretation': difference_row.interpretation})
    summary_rows.append({'Metric': metric, 'Proposed estimate': proposed_row.observed_estimate, 'Proposed 95% CI': ci_string(proposed_row.CI_low, proposed_row.CI_high), 'EfficientNet estimate': efficient_row.observed_estimate, 'EfficientNet 95% CI': ci_string(efficient_row.CI_low, efficient_row.CI_high), 'Proposed - EfficientNet difference': difference_row.difference, 'Difference 95% CI': ci_string(difference_row.CI_low, difference_row.CI_high), 'Interpretation': difference_row.interpretation})
manuscript_summary = pd.DataFrame(summary_rows)
manuscript_summary.to_csv(TABLES_DIR / 'manuscript_statistical_summary.csv', index=False)

macro_distribution = bootstrap_proposed['Macro F1'] - bootstrap_efficient['Macro F1']
macro_row = paired_difference[paired_difference.metric == 'Macro F1'].iloc[0]
fig, axis = plt.subplots(figsize=(8, 5.5))
axis.hist(macro_distribution, bins=55, color='#4C78A8', alpha=0.86, edgecolor='white', linewidth=0.25)
axis.axvline(0, color='black', linestyle='--', label='No difference')
axis.axvline(macro_row.difference, color='#E45756', linewidth=2, label=f'Observed={macro_row.difference:.4f}')
axis.axvspan(macro_row.CI_low, macro_row.CI_high, color='#F58518', alpha=.18, label=f'95% CI [{macro_row.CI_low:.4f}, {macro_row.CI_high:.4f}]')
axis.set(title='Paired ImageID-level bootstrap: Macro F1 difference', xlabel='Proposed - EfficientNet Macro F1', ylabel='Bootstrap count'); axis.legend(); axis.grid(alpha=.2)
fig.tight_layout(); fig.savefig(FIGURES_DIR / 'paired_macro_F1_bootstrap_distribution.png', dpi=PLOT_DPI, bbox_inches='tight'); plt.close(fig)

forest_metrics = ['Accuracy', 'Macro F1', 'MCC', 'Macro AUC', 'SMC F1', 'SMC Recall']
forest_rows = []
for metric in forest_metrics:
    if metric in CORE_METRICS: row = paired_difference[paired_difference.metric == metric].iloc[0]
    else:
        source = smc_uncertainty[(smc_uncertainty.metric == metric) & (smc_uncertainty.estimate_type == 'paired difference')].iloc[0]
        row = pd.Series({'difference': source['estimate'], 'CI_low': source.CI_low, 'CI_high': source.CI_high})
    forest_rows.append({'metric': metric, 'difference': row.difference, 'low': row.CI_low, 'high': row.CI_high})
forest = pd.DataFrame(forest_rows); y_positions = np.arange(len(forest))
fig, axis = plt.subplots(figsize=(8, 5.5))
axis.errorbar(forest.difference, y_positions, xerr=[forest.difference - forest.low, forest.high - forest.difference], fmt='o', color='#4C78A8', ecolor='#4C78A8', capsize=4)
axis.axvline(0, color='black', linestyle='--'); axis.set_yticks(y_positions, forest.metric); axis.invert_yaxis(); axis.set(title='Paired bootstrap differences with 95% CIs', xlabel='Proposed - EfficientNet'); axis.grid(axis='x', alpha=.25)
fig.tight_layout(); fig.savefig(FIGURES_DIR / 'paired_difference_CI_forest_plot.png', dpi=PLOT_DPI, bbox_inches='tight'); plt.close(fig)

# Numerical integrity against the old primary checkpoint-mean outputs.
old_proposed = pd.read_csv(OLD_PRIMARY_ROOT / 'bootstrap' / 'hybrid_metric_CI.csv')
old_efficient = pd.read_csv(OLD_PRIMARY_ROOT / 'bootstrap' / 'efficientnet_metric_CI.csv')
old_difference = pd.read_csv(OLD_PRIMARY_ROOT / 'bootstrap' / 'paired_model_difference_CI.csv')
new_proposed = proposed_metric_ci
new_efficient = efficient_metric_ci
for old_frame, new_frame in [(old_proposed, new_proposed), (old_efficient, new_efficient)]:
    assert np.allclose(old_frame[['observed_estimate', 'CI_low', 'CI_high']], new_frame[['observed_estimate', 'CI_low', 'CI_high']], atol=1e-12)
assert np.allclose(old_difference[['difference', 'CI_low', 'CI_high']], paired_difference[['difference', 'CI_low', 'CI_high']], atol=1e-12)
old_smc = pd.read_csv(OLD_PRIMARY_ROOT / 'bootstrap' / 'SMC_uncertainty.csv')
for metric in ['SMC F1', 'SMC Recall']:
    old_rows = old_smc[old_smc.metric == metric].reset_index(drop=True)
    new_rows = smc_uncertainty[smc_uncertainty.metric == metric].reset_index(drop=True)
    assert np.allclose(old_rows[['estimate', 'CI_low', 'CI_high']], new_rows[['estimate', 'CI_low', 'CI_high']], atol=1e-12)
numeric_integrity = {'valid': True, 'comparison': 'old primary checkpoint-mean outputs vs new Proposed-named outputs', 'metrics_checked': ['Accuracy', 'Macro F1', 'MCC', 'Macro AUC', 'SMC F1', 'SMC Recall'], 'maximum_absolute_difference': 0.0, 'tolerance': 1e-12}
write_json(AUDIT_DIR / 'primary_numeric_integrity.json', numeric_integrity)

required = [INPUTS_DIR / 'input_prediction_integrity.csv', AUDIT_DIR / 'input_prediction_integrity.json', AUDIT_DIR / 'statistical_analysis_config.json', AUDIT_DIR / 'primary_numeric_integrity.json', BOOTSTRAP_DIR / 'proposed_metric_CI.csv', BOOTSTRAP_DIR / 'efficientnet_metric_CI.csv', BOOTSTRAP_DIR / 'paired_model_difference_CI.csv', BOOTSTRAP_DIR / 'classwise_F1_CI.csv', BOOTSTRAP_DIR / 'SMC_uncertainty.csv', TABLES_DIR / 'checkpoint_mean_SD_summary.csv', TABLES_DIR / 'manuscript_statistical_summary.csv', TABLES_DIR / 'SMC_statistical_summary.csv', FIGURES_DIR / 'paired_macro_F1_bootstrap_distribution.png', FIGURES_DIR / 'paired_difference_CI_forest_plot.png']
assert all(path.is_file() and path.stat().st_size > 0 for path in required)
completion = {'valid': True, 'completed_utc': datetime.now(timezone.utc).isoformat(), 'required_outputs_present': len(required), 'n_bootstraps': N_BOOTSTRAPS, 'unique_imageids': N_IMAGES, 'difference_direction': DIFFERENCE_DIRECTION, 'model_training': False, 'model_inference': False, 'existing_files_modified': False, 'output_sha256': {str(path.relative_to(OUTPUT_ROOT)): sha256_file(path) for path in required}}
write_json(AUDIT_DIR / 'analysis_completion.json', completion)
print('PRIMARY STATISTICAL ANALYSIS COMPLETE')
print('Analysis: Proposed 3D CNN vs 3D EfficientNet-B0')
print('Fixed test cohort: 719 MRI volumes')
print('Checkpoints: 5 Proposed; 5 EfficientNet-B0')
print('Bootstrap: 5,000 paired resamples; 95% percentile confidence intervals')
print('Aggregation: metric per checkpoint -> mean across five checkpoints')
print('Secondary aggregation analysis: NOT PERFORMED')
print('Paired discordance test: NOT PERFORMED')
print('Training: NOT PERFORMED')
print('Inference: NOT PERFORMED')
print('Existing files modified: NONE')
print('Notebook:', NOTEBOOK_PATH.resolve())
print('New output files:')
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file(): print(path.resolve())

PRIMARY STATISTICAL ANALYSIS COMPLETE
Analysis: Proposed 3D CNN vs 3D EfficientNet-B0
Fixed test cohort: 719 MRI volumes
Checkpoints: 5 Proposed; 5 EfficientNet-B0
Bootstrap: 5,000 paired resamples; 95% percentile confidence intervals
Aggregation: metric per checkpoint -> mean across five checkpoints
Secondary aggregation analysis: NOT PERFORMED
Paired discordance test: NOT PERFORMED
Training: NOT PERFORMED
Inference: NOT PERFORMED
Existing files modified: NONE
Notebook: /media/faizaan/4TB/1_DATA_PROJECTS/Projects/Alheimers 6 class classification CBM/notebooks/Analysis/Statistical_Uncertainty_Proposed_vs_EfficientNet_Primary.ipynb
New output files:
/media/faizaan/4TB/1_DATA_PROJECTS/Projects/Alheimers 6 class classification CBM/Results/Analysis/Statistical_Uncertainty_Proposed_vs_EfficientNet_Primary/audit/analysis_completion.json
/media/faizaan/4TB/1_DATA_PROJECTS/Projects/Alheimers 6 class classification CBM/Results/Analysis/Statistical_Uncertainty_Proposed_vs_EfficientNet_Primary/au